# Dispersion of cylindrical TM modes in a magnetized plasma column

A dispersion relation connects a wave frequency to its wavenumber. In an unbounded uniform plasma, many useful dispersion relations have compact analytical forms. Laboratory plasmas, however, are often confined by finite
geometries and magnetic fields, so the oscillations are no longer arbitrary plane waves but discrete modes determined by both the plasma response and the geometry of the system.

This notebook studies transverse magnetic (TM) modes in a cylindrical plasma column. The cylindrical boundary quantizes the radial structure into discrete normal modes, analogous to the standing-wave patterns of a vibrating string or a drumhead. The goal is to compare how different plasma physical models modify wave propagation while preserving the same geometry:

- An empty cylindrical waveguide, which establishes the purely electromagnetic behavior of the geometry in the absence of plasma effects.

- A cold-fluid plasma model, where electrons respond to the wave but have no thermal motion.

- A warm-fluid plasma model, which adds electron pressure through the Debye length. This demonstrates how finite temperatures shift the wave branches relative to the cold result.

- A kinetic formulation, which includes the full velocity-space response of the electrons, so that electrons near the wave's phase velocity can resonantly exchange energy with it, an effect called Landau damping.

The empty guide, cold and warm-fluid cases have analytical solutions, so they provide controlled benchmarks for the numerical methods. The analysis then extends to the kinetic problem, where the dispersion relation generally requires root finding and where complex frequencies carry information about wave damping. The result is a compact demonstration of how PlasmaPy can combine physical units, special functions, and numerical solvers in a reproducible plasma-wave study.

Throughout the notebook, the plasma is assumed to be fully ionized and collisionless, with stationary ions providing a neutralizing background. Extensions involving neutral collisions or more general collision operators are beyond the scope of the models considered here, so it is discussed only as a boundary of applicability rather than
implemented without a cited dispersion relation.


## Physical setting

We consider a homogeneous cylindrical plasma column of radius $a$, as illustrated below. A uniform external magnetic field $B_0 \hat{z}$ is applied along the cylinder axis, confining the plasma and making the electron motion the relevant wave-supporting response for the TM modes considered here. The ions provide a stationary neutralizing background.

![model_geometry.png](model_geometry.png)

The waves studied in this notebook propagate along the cylinder axis with longitudinal wavenumber $k$. Because the plasma occupies a finite cylindrical domain, the radial structure of the fields cannot be arbitrary. Instead, the electromagnetic fields must satisfy boundary conditions at the cylinder wall, leading to a discrete set of allowed transverse modes. As a result, each TM is identified by two integers $(m, n)$. $m$ is the azimuthal mode number, describing the angular variation around the cylinder, and $n$ is the radial mode number.

The radial dependence of the fields is described by Bessel functions $J_m$. Imposing the boundary condition at the plasma edge selects only specific radial eigenvalues  $X_{mn}$, defined by

$$
J_m(X_{mn}) = 0.
$$

The quantity $X_{mn}$ therefore determines the transverse structure of the $(m,n)$-th mode and enters directly into the dispersion relations studied throughout this notebook.

The cold and warm-fluid dispersion relations follow the treatment of space-charge waves in section 4.8 of Krall & Trivelpiece, *Principles of Plasma Physics* (1973), especially Eqs. 4.8.15 and 4.8.16. The same class of bounded-plasma eigenvalue problems is discussed in
[Trivelpiece & Gould (1959)](https://authors.library.caltech.edu/records/hfhkd-mm296).

## Scope and normalization

The notebook develops the following sequence:

1. Define a normalized cylindrical TM dispersion problem.
2. Compute physically meaningful reference scales with PlasmaPy.
3. Evaluate the analytical cold-plasma and warm-fluid branches.
4. Obtain the same branches numerically using root-finding methods.
5. Use the validated numerical approach to extend the analysis to the kinetic model, where the dispersion function $Q_m$ must be solved numerically.

To facilitate comparisons between the different models, all quantities are expressed in dimensionless form. Frequencies are normalized to the electron plasma frequency $\omega_{pe}$, while lengths are normalized to the cylinder radius $a$. The resulting dimensionless variables are

$$
\bar{\omega} = \frac{\omega}{\omega_{pe}},
\qquad
\bar{k} = ka,
\qquad
\bar{c} = \frac{c}{a\omega_{pe}},
\qquad
\bar{\lambda}_D = \frac{\lambda_D}{a},
$$

Where $\lambda_D$ is the electron Debye length. PlasmaPy is used here because these reference scales carry units, depend on particle properties, and are common sources of mistakes in dispersion calculations.


In [ ]:
from typing import Literal

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.constants import c
from numpy.typing import ArrayLike, NDArray
from scipy.optimize import brentq, root
from scipy.special import jn_zeros

from plasmapy.dispersion import plasma_dispersion_func_deriv
from plasmapy.formulary.frequencies import plasma_frequency
from plasmapy.formulary.lengths import Debye_length

Branch = Literal["low", "high"]
BRANCHES: tuple[Branch, Branch] = ("low", "high")
FloatArray = NDArray[np.float64]
ComplexArray = NDArray[np.complex128]

plt.rcParams.update(
    {
        "figure.figsize": (8.0, 5.0),
        "axes.grid": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "font.size": 11,
        "legend.frameon": False,
        "lines.linewidth": 2.2,
    }
)

COLORS = {
    (0, 1): "#0072B2",
    (1, 1): "#D55E00",
    (2, 1): "#009E73",
}

## Reference parameters

The numerical values below are representative of laboratory plasma-column
conditions. They set the dimensional conversion between normalized
quantities and physical frequencies or wavenumbers; the dispersion
relations themselves are solved in normalized form.

In [ ]:
n_e = (2.2e8 * u.cm**-3).to(u.m**-3)
T_e = 10 * u.eV
a = (5.2 * u.cm).to(u.m)

omega_pe = plasma_frequency(n_e, particle="e-")
f_pe = plasma_frequency(n_e, particle="e-", to_hz=True)
lambda_D = Debye_length(T_e.to(u.K, equivalencies=u.temperature_energy()), n_e)

cbar = c.to_value(u.m / u.s) / (a.to_value(u.m) * omega_pe.to_value(u.rad / u.s))
lambda_D_bar = (lambda_D / a).to_value(u.dimensionless_unscaled)

## 1. Cold-fluid plasma dispersion relation

In the cold limit, the normalized TM frequencies can be obtained analytically through the algebraic solution of Eq. 4.8.15 in section 4.8, "Space Charge Waves," of Krall & Trivelpiece. In normalized form, the cold plasma dispersion function can be written as

$$
R_c(\bar\omega, \bar k)
=
\bar{k}^2 - \frac{\bar\omega^2}{\bar c^2}
+ \frac{X_{mn}^2}{\epsilon_c(\bar\omega)} = 0,
$$

where

$$
\epsilon_c(\bar\omega) = 1 - \frac{1}{\bar\omega^2}
$$

is the cold electron dielectric function. Expanding the dispersion relation into a polynomial in $y = \bar\omega^2$ gives

$$
y^2
-
\left[1 + \bar c^2\left(\bar k^2 + X_{mn}^2\right)\right] y
+
\bar c^2 \bar k^2 = 0.
$$

The two analytical branches are therefore

$$
\bar\omega_{\pm}
=
\left\{
\frac{1}{2}
\left[
1 + \bar c^2(\bar k^2 + X_{mn}^2)
\pm
\sqrt{
\left(1 + \bar c^2(\bar k^2 + X_{mn}^2)\right)^2
- 4\bar c^2\bar k^2
}
\right]
\right\}^{1/2}.
$$

Although this problem does not require a numerical solution, in general dispersion relations do. We therefore solve the cold plasma relation numerically and compare the resulting roots with the analytical solution before moving on to more complex dispersion models.


## 2. Warm-fluid dispersion relation

The warm-fluid extension incorporates electron pressure through the
Debye length. The dielectric response becomes

$$
\epsilon_w(\bar\omega, \bar k)
=
1 - \frac{1}{\bar\omega^2 - 3\bar\lambda_D^2\bar k^2},
$$

and the dispersion relation is given by the solution of the following expression:

$$
\bar{k}^2 - \frac{\bar\omega^2}{\bar c^2}
+ \frac{X_{mn}^2}{\epsilon_w(\bar\omega, \bar k)} = 0.
$$

Following Eq. 4.8.16 in Krall & Trivelpiece, this dispersion relation again reduces
to a quadratic polynomial in $y = \bar\omega^2$. Define

$$
D = 3\bar\lambda_D^2\bar k^2,
$$

$$
\alpha = 1 + D + \bar c^2(\bar k^2 + X_{mn}^2),
$$

and

$$
\beta = \bar c^2\left[\bar k^2(1 + D) + X_{mn}^2D\right].
$$

Then

$$
\bar\omega_{\pm}^2
=
\frac{1}{2}
\left(
\alpha \pm \sqrt{\alpha^2 - 4\beta}
\right).
$$

The factor $D$ measures the size of the pressure correction. Since it is
proportional to $(k\lambda_D)^2$, the warm-fluid branch must approach the
cold branch when $k\lambda_D \ll 1$ and should deviate most clearly at
larger axial wavenumber.

For reference, we use thermal parameters often used in experiments
to quantify the size of the exact warm-fluid correction.

In [ ]:
def bessel_zero(m: int, n: int) -> float:
    """Return the n-th positive zero of the Bessel function J_m."""
    if m < 0 or n < 1:
        msg = "m must be nonnegative and n must be at least 1."
        raise ValueError(msg)
    return float(jn_zeros(m, n)[-1])


def _as_float_array(values: ArrayLike) -> FloatArray:
    """Return values as a floating NumPy array."""
    return np.asarray(values, dtype=float)


def _select_branch(y_low: FloatArray, y_high: FloatArray, branch: Branch) -> FloatArray:
    """Select the requested branch of omega_bar squared."""
    if branch == "low":
        return y_low
    if branch == "high":
        return y_high
    msg = "branch must be 'low' or 'high'."
    raise ValueError(msg)


def cold_frequency(
    kbar: ArrayLike, cbar: float, x_mn: float, branch: Branch
) -> FloatArray:
    """Analytical cold-plasma normalized angular frequency, omega / omega_pe."""
    kbar = _as_float_array(kbar)
    alpha = 1 + cbar**2 * (kbar**2 + x_mn**2)
    discriminant = np.maximum(alpha**2 - 4 * cbar**2 * kbar**2, 0.0)
    root_discriminant = np.sqrt(discriminant)
    y_low = 0.5 * (alpha - root_discriminant)
    y_high = 0.5 * (alpha + root_discriminant)
    return np.sqrt(np.maximum(_select_branch(y_low, y_high, branch), 0.0))


def warm_frequency(
    kbar: ArrayLike,
    cbar: float,
    x_mn: float,
    lambda_d_bar: float,
    branch: Branch,
) -> FloatArray:
    """Analytical warm-fluid normalized angular frequency, omega / omega_pe."""
    kbar = _as_float_array(kbar)
    thermal_shift = 3 * lambda_d_bar**2 * kbar**2
    alpha = 1 + thermal_shift + cbar**2 * (kbar**2 + x_mn**2)
    beta = cbar**2 * (kbar**2 * (1 + thermal_shift) + x_mn**2 * thermal_shift)
    discriminant = np.maximum(alpha**2 - 4 * beta, 0.0)
    root_discriminant = np.sqrt(discriminant)
    y_low = 0.5 * (alpha - root_discriminant)
    y_high = 0.5 * (alpha + root_discriminant)
    return np.sqrt(np.maximum(_select_branch(y_low, y_high, branch), 0.0))


def cold_dielectric(omega_bar: ArrayLike) -> ComplexArray:
    """Cold electron dielectric function epsilon_c."""
    omega_bar = np.asarray(omega_bar, dtype=complex)
    return 1 - 1 / omega_bar**2


def warm_dielectric(
    omega_bar: ArrayLike,
    kbar: ArrayLike,
    lambda_d_bar: float,
) -> ComplexArray:
    """Warm-fluid electron dielectric function epsilon_w."""
    omega_bar = np.asarray(omega_bar, dtype=complex)
    kbar = np.asarray(kbar, dtype=float)
    return 1 - 1 / (omega_bar**2 - 3 * lambda_d_bar**2 * kbar**2)


def cold_dispersion(omega_bar: float, kbar: float, cbar: float, x_mn: float) -> float:
    """Cold-plasma dispersion function R_c for real frequencies."""
    dispersion = kbar**2 - omega_bar**2 / cbar**2 + x_mn**2 / cold_dielectric(omega_bar)
    return float(np.real(dispersion))


def warm_dispersion(
    omega_bar: float,
    kbar: float,
    cbar: float,
    x_mn: float,
    lambda_d_bar: float,
) -> float:
    """Warm-fluid dispersion function R_w for real frequencies."""
    dispersion = (
        kbar**2
        - omega_bar**2 / cbar**2
        + x_mn**2 / warm_dielectric(omega_bar, kbar, lambda_d_bar)
    )
    return float(np.real(dispersion))


def _real_root_in_interval(
    dispersion_function,
    left: float,
    right: float,
    *,
    samples: int = 800,
    xtol: float = 1e-12,
    rtol: float = 1e-12,
) -> float:
    """Locate one real root by scanning for a bracket and refining it."""
    if not right > left:
        msg = f"Invalid root interval: ({left}, {right})."
        raise ValueError(msg)

    xs = np.linspace(left, right, samples)
    ys = np.array([dispersion_function(x) for x in xs], dtype=float)
    finite = np.isfinite(ys) & (np.abs(ys) < 1e12)
    xs = xs[finite]
    ys = ys[finite]

    for lower, upper, y_lower, y_upper in zip(
        xs[:-1], xs[1:], ys[:-1], ys[1:], strict=True
    ):
        if y_lower == 0:
            return float(lower)
        if np.signbit(y_lower) != np.signbit(y_upper):
            return float(
                brentq(dispersion_function, lower, upper, xtol=xtol, rtol=rtol)
            )

    msg = f"No sign-changing bracket found in ({left}, {right})."
    raise RuntimeError(msg)


def cold_root(kbar: float, cbar: float, x_mn: float, branch: Branch) -> float:
    """Recover a cold branch numerically by solving the dispersion relation."""
    if np.isclose(kbar, 0.0) and branch == "low":
        return 0.0

    resonance = 1.0
    upper = 1.1 * cold_frequency(kbar, cbar, x_mn, "high").item() + 0.5

    def dispersion_function(omega: float) -> float:
        return cold_dispersion(omega, kbar, cbar, x_mn)

    if branch == "low":
        return _real_root_in_interval(dispersion_function, 1e-8, resonance - 1e-6)
    return _real_root_in_interval(dispersion_function, resonance + 1e-6, upper)


def warm_root(
    kbar: float,
    cbar: float,
    x_mn: float,
    lambda_d_bar: float,
    branch: Branch,
) -> float:
    """Recover a warm-fluid branch numerically by solving the dispersion relation."""
    if np.isclose(kbar, 0.0):
        return warm_frequency(kbar, cbar, x_mn, lambda_d_bar, branch).item()

    resonance = np.sqrt(1 + 3 * lambda_d_bar**2 * kbar**2)
    upper = 1.1 * warm_frequency(kbar, cbar, x_mn, lambda_d_bar, "high").item() + 0.5

    def dispersion_function(omega: float) -> float:
        return warm_dispersion(omega, kbar, cbar, x_mn, lambda_d_bar)

    if branch == "low":
        return _real_root_in_interval(dispersion_function, 1e-8, resonance - 1e-6)
    return _real_root_in_interval(dispersion_function, resonance + 1e-6, upper)


def kinetic_dispersion_qm(
    omega_bar: complex,
    kbar: float,
    cbar: float,
    x_mn: float,
    lambda_d_bar: float,
) -> complex:
    """Kinetic dispersion function Q_m for bounded TM modes."""
    if kbar <= 0:
        msg = "The kinetic dispersion function is singular at kbar = 0."
        raise ValueError(msg)

    field_factor = (cbar**2 * kbar**2 - omega_bar**2) / (
        cbar**2 * (kbar**2 + x_mn**2) - omega_bar**2
    )
    zeta = omega_bar / (np.sqrt(2) * kbar * lambda_d_bar)
    kinetic_susceptibility = plasma_dispersion_func_deriv(zeta) / (
        2 * lambda_d_bar**2 * kbar**2
    )
    return field_factor * kinetic_susceptibility - 1


def kinetic_root(
    kbar: float,
    cbar: float,
    x_mn: float,
    lambda_d_bar: float,
    initial_guess: complex,
    *,
    tolerance: float = 1e-10,
) -> complex:
    """Solve Q_m = 0 for a complex frequency using a real-valued system."""

    def system(values: FloatArray) -> FloatArray:
        omega = values[0] + 1j * values[1]
        dispersion = kinetic_dispersion_qm(omega, kbar, cbar, x_mn, lambda_d_bar)
        return np.array([dispersion.real, dispersion.imag])

    solution = root(
        system,
        np.array([initial_guess.real, initial_guess.imag]),
        method="hybr",
        options={"xtol": tolerance, "maxfev": 300},
    )
    if not solution.success or np.linalg.norm(system(solution.x)) > 1e-7:
        msg = f"Kinetic root failed at kbar={kbar:.3g}: {solution.message}"
        raise RuntimeError(msg)
    return complex(solution.x[0], solution.x[1])


def frequency_hz(omega_bar_values: ArrayLike) -> FloatArray:
    """Convert normalized angular frequency to ordinary frequency in Hz."""
    return np.asarray(omega_bar_values, dtype=float) * f_pe.to_value(u.Hz)


def wavenumber_per_meter(kbar_values: ArrayLike) -> FloatArray:
    """Convert normalized axial wavenumber to inverse meters."""
    return np.asarray(kbar_values, dtype=float) / a.to_value(u.m)

## 3. Vacuum and neutral-gas reference

A useful reference is the limit with no mobile electron plasma response.
A neutral gas without ionization would not support the space-charge wave
mode described by the electron dielectric function. In the collisionless
model used here, the closest comparison is therefore an empty cylindrical
waveguide, whose TM frequency is

$$
\bar\omega_{\mathrm{vac}} = \bar c\sqrt{\bar k^2 + X_{mn}^2}.
$$

This comparison clarifies the core plasma effect. The high-frequency
branch connects to an electromagnetic waveguide-like mode, while the
low-frequency branch is the plasma-supported space-charge mode and has
no neutral-gas analogue in this model.

In [ ]:
def vacuum_frequency(kbar: ArrayLike, cbar: float, x_mn: float) -> FloatArray:
    """Normalized angular frequency for an empty cylindrical waveguide."""
    kbar = _as_float_array(kbar)
    return cbar * np.sqrt(kbar**2 + x_mn**2)


kbar_grid = np.linspace(0, 10, 300)
k_grid = wavenumber_per_meter(kbar_grid)
reference_mode = (0, 1)
x_reference = bessel_zero(*reference_mode)

cold_low_reference = cold_frequency(kbar_grid, cbar, x_reference, "low")
cold_high_reference = cold_frequency(kbar_grid, cbar, x_reference, "high")
warm_low_reference = warm_frequency(kbar_grid, cbar, x_reference, lambda_D_bar, "low")
vacuum_reference = vacuum_frequency(kbar_grid, cbar, x_reference)

fig, axes = plt.subplots(ncols=2, figsize=(12, 4.6), sharex=True)
axes[0].plot(
    k_grid,
    frequency_hz(vacuum_reference),
    color="0.30",
    linestyle=":",
    label="empty waveguide",
)
axes[0].plot(
    k_grid, frequency_hz(cold_high_reference), color="#0072B2", label="cold high branch"
)
axes[0].plot(
    k_grid,
    frequency_hz(cold_low_reference),
    color="#D55E00",
    label="cold space-charge branch",
)
axes[0].plot(
    k_grid,
    frequency_hz(warm_low_reference),
    color="#009E73",
    linestyle="--",
    label="warm low branch",
)
axes[0].set_title(r"TM$_{0,1}$ full frequency scale")
axes[0].set_xlabel(r"$k$ [m$^{-1}$]")
axes[0].set_ylabel("frequency [Hz]")
axes[0].legend(fontsize=8)

axes[1].plot(
    k_grid, frequency_hz(cold_low_reference), color="#D55E00", label="cold low"
)
axes[1].plot(
    k_grid,
    frequency_hz(warm_low_reference),
    color="#009E73",
    linestyle="--",
    label="warm low",
)
axes[1].set_title("Space-charge branch zoom")
axes[1].set_xlabel(r"$k$ [m$^{-1}$]")
axes[1].set_ylabel("frequency [Hz]")
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()

## 4. Cold cylindrical TM spectrum

The first comparison shows how the Bessel-root labels organize the cold
spectrum. Increasing $m$ or $n$ increases $X_{mn}$, which raises the
high-frequency cutoff and changes the low-frequency branch through the
same boundary-selected radial scale.

In [ ]:
kbar_grid = np.linspace(0, 10, 300)
k_grid = wavenumber_per_meter(kbar_grid)
mode_grid = [(m, n) for m in range(3) for n in range(1, 4)]

fig, axes = plt.subplots(ncols=2, figsize=(12, 4.6), sharex=True)
for branch, ax in zip(BRANCHES, axes, strict=True):
    for m, n in mode_grid:
        x_mn = bessel_zero(m, n)
        omega = cold_frequency(kbar_grid, cbar, x_mn, branch)
        ax.plot(k_grid, frequency_hz(omega), label=rf"TM$_{{{m},{n}}}$")
    ax.set_title(f"Cold {branch}-frequency branch")
    ax.set_xlabel(r"$k$ [m$^{-1}$]")
    ax.set_ylabel("frequency [Hz]")

axes[0].legend(ncols=3, fontsize=8)
fig.tight_layout()
plt.show()

## 5. Numerical recovery of the cold branches

The cold plasma dispersion relation is now solved directly. The solid curves are the analytical expressions, while the markers are numerical roots. Agreement shows that the root-finding setup and branch intervals recover the known cold-plasma solution.


In [ ]:
selected_modes = [(0, 1), (1, 1), (2, 1)]
kbar_roots = np.linspace(0, 10, 90)
k_roots = wavenumber_per_meter(kbar_roots)

fig, axes = plt.subplots(ncols=2, figsize=(12, 4.6), sharex=True)
cold_error_rows = []

for branch, marker, ax in zip(BRANCHES, ["o", "s"], axes, strict=True):
    for mode in selected_modes:
        x_mn = bessel_zero(*mode)
        color = COLORS[mode]
        analytical = cold_frequency(kbar_roots, cbar, x_mn, branch)
        numerical = np.array([cold_root(kb, cbar, x_mn, branch) for kb in kbar_roots])
        relative_error = np.abs(analytical - numerical) / np.maximum(
            np.abs(analytical), 1e-12
        )

        cold_error_rows.append(
            (
                f"TM({mode[0]},{mode[1]})",
                branch,
                relative_error.max(),
                relative_error.mean(),
            )
        )
        ax.plot(
            k_roots,
            frequency_hz(analytical),
            color=color,
            label=rf"TM$_{{{mode[0]},{mode[1]}}}$",
        )
        ax.scatter(
            k_roots[::5],
            frequency_hz(numerical[::5]),
            color=color,
            marker=marker,
            s=24,
            edgecolor="white",
            linewidth=0.5,
            zorder=3,
        )
    ax.set_title(f"Cold {branch} branch: analytical and numerical roots")
    ax.set_xlabel(r"$k$ [m$^{-1}$]")
    ax.set_ylabel("frequency [Hz]")

axes[0].legend(fontsize=9)
fig.tight_layout()
plt.show()

max_cold_relative_error = max(row[2] for row in cold_error_rows)
if max_cold_relative_error >= 1e-9:
    msg = f"Cold case dispersion function recovery exceeded tolerance: {max_cold_relative_error:.3e}."
    raise RuntimeError(msg)

The analytical cold-plasma dispersion relation provides a direct benchmark for the numerical root-finding procedure. For each mode and branch, the relative error is defined as

$$ \epsilon_{\mathrm{rel}} = \frac{ \left|\bar{\omega}_{\mathrm{num}} - \bar{\omega}_{\mathrm{ana}}\right|}{\left|
\bar{\omega}_{\mathrm{ana}}\right|}$$


The table below summarizes the maximum and mean relative errors across the entire wavenumber range.

In [ ]:
import pandas as pd

cold_error_table = pd.DataFrame(
    cold_error_rows,
    columns=[
        "Mode",
        "Branch",
        "Max relative error",
        "Mean relative error",
    ],
)

cold_error_table.style.format(
    {
        "Max relative error": "{:.3e}",
        "Mean relative error": "{:.3e}",
    }
)

## 6. Warm-fluid thermal correction

The warm-fluid model separates two questions. First, how much do thermal
corrections move the branch relative to the cold plasma result? Second,
can the numerical solver recover the exact warm-fluid branch as accurately as it did in the cold benchmark?

The plots below use the exact warm-fluid analytical solution. The
dimensionless parameter $k\lambda_D = \bar k\bar\lambda_D$ is shown
through the fractional shift so that the thermal correction is
interpretable independently of the chosen dimensional units.

In [ ]:
kbar_thermal = np.geomspace(0.02, 10, 300)
k_thermal = wavenumber_per_meter(kbar_thermal)

fig, axes = plt.subplots(ncols=2, figsize=(12, 4.6))
for mode in selected_modes:
    x_mn = bessel_zero(*mode)
    color = COLORS[mode]
    cold_low = cold_frequency(kbar_thermal, cbar, x_mn, "low")
    warm_low = warm_frequency(kbar_thermal, cbar, x_mn, lambda_D_bar, "low")
    fractional_shift_percent = 100 * (warm_low - cold_low) / cold_low

    axes[0].plot(
        k_thermal, frequency_hz(cold_low), color=color, linestyle="--", alpha=0.55
    )
    axes[0].plot(
        k_thermal,
        frequency_hz(warm_low),
        color=color,
        label=rf"TM$_{{{mode[0]},{mode[1]}}}$",
    )
    axes[1].plot(
        kbar_thermal,
        fractional_shift_percent,
        color=color,
        label=rf"TM$_{{{mode[0]},{mode[1]}}}$",
    )

axes[0].set_title("Low branch: cold baseline and warm-fluid result")
axes[0].set_xlabel(r"$k$ [m$^{-1}$]")
axes[0].set_ylabel("frequency [Hz]")
axes[0].legend(fontsize=9)

axes[1].set_title("Fractional thermal shift of the low branch")
axes[1].set_xlabel(r"$\bar{k} = ka$")
axes[1].set_ylabel(r"$100(\bar\omega_w - \bar\omega_c) / \bar\omega_c$ [%]")
axes[1].legend(fontsize=9)
axes[1].text(
    0.03,
    0.95,
    rf"max $k\lambda_D$ = {kbar_thermal[-1] * lambda_D_bar:.3f}",
    transform=axes[1].transAxes,
    va="top",
    fontsize=9,
)

fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 4.6), sharex=True)
warm_error_rows = []

for branch, marker, ax in zip(BRANCHES, ["o", "s"], axes, strict=True):
    for mode in selected_modes:
        x_mn = bessel_zero(*mode)
        color = COLORS[mode]
        analytical = warm_frequency(kbar_roots, cbar, x_mn, lambda_D_bar, branch)
        numerical = np.array(
            [warm_root(kb, cbar, x_mn, lambda_D_bar, branch) for kb in kbar_roots]
        )
        relative_error = np.abs(analytical - numerical) / np.maximum(
            np.abs(analytical), 1e-12
        )
        warm_error_rows.append(
            (
                f"TM({mode[0]},{mode[1]})",
                branch,
                relative_error.max(),
                relative_error.mean(),
            )
        )

        ax.plot(
            k_roots,
            frequency_hz(analytical),
            color=color,
            label=rf"TM$_{{{mode[0]},{mode[1]}}}$",
        )
        ax.scatter(
            k_roots[::5],
            frequency_hz(numerical[::5]),
            color=color,
            marker=marker,
            s=24,
            edgecolor="white",
            linewidth=0.5,
            zorder=3,
        )
    ax.set_title(f"Warm-fluid {branch} branch: analytical and numerical roots")
    ax.set_xlabel(r"$k$ [m$^{-1}$]")
    ax.set_ylabel("frequency [Hz]")

axes[0].legend(fontsize=9)
fig.tight_layout()
plt.show()

max_warm_relative_error = max(row[2] for row in warm_error_rows)
if max_warm_relative_error >= 1e-9:
    msg = f"Warm-fluid dispersion function recovery exceeded tolerance: {max_warm_relative_error:.3e}."
    raise RuntimeError(msg)

The warm-fluid dispersion relation provides a more demanding benchmark because the dielectric response includes thermal corrections through the Debye length. The same relative-error metric is used to compare the analytical and numerical roots.

In [ ]:
warm_error_table = pd.DataFrame(
    warm_error_rows,
    columns=[
        "Mode",
        "Branch",
        "Max relative error",
        "Mean relative error",
    ],
)

warm_error_table.style.format(
    {
        "Max relative error": "{:.3e}",
        "Mean relative error": "{:.3e}",
    }
)

In [ ]:
print(
    f"Maximum cold-plasma relative error: {max(row[2] for row in cold_error_rows):.3e}"
)

print(
    f"Maximum warm-fluid relative error: {max(row[2] for row in warm_error_rows):.3e}"
)

The numerical roots reproduce the analytical dispersion relations with relative errors at the level reported above. This agreement validates the root-finding procedure before applying it to the kinetic
dispersion relation, where analytical solutions are generally unavailable.

## 7. Kinetic dispersion relation

The fluid models compress the electron response into dielectric functions with real-valued branches for the parameters used above. A kinetic description replaces that closure with a velocity-space response. The plasma dispersion function enters because particles near the parallel phase velocity can exchange energy resonantly with the wave, so the mode frequency is generally complex. In this convention, a negative imaginary frequency corresponds to Landau damping.

For the bounded plasma column, the kinetic dispersion relation is expressed through the dispersion function

$$
Q_m(\bar\omega, \bar k) =
\frac{\bar c^2\bar k^2 - \bar\omega^2}
{\bar c^2(\bar k^2 + X_{mn}^2) - \bar\omega^2}
\frac{Z'(\zeta)}{2\bar\lambda_D^2\bar k^2}
- 1,
$$

with

$$
\zeta = \frac{\bar\omega}{\sqrt{2}\,\bar k\bar\lambda_D}.
$$

This section follows the strategy used by Hoyos and coauthors for bounded
plasma eigenvalue problems: H. Hoyos, Jaime and A. Rodriguez, Carlos,
["Numerical Solution of an Eigenvalue Problem for Bounded Plasma"](http://www.scielo.org.co/scielo.php?script=sci_arttext&pid=S1692-33242011000200017),
*Revista Ingeniería Universidad de Medellín* **10**, 179-188 (2011), and the
related Journal of Physics: Conference Series papers by
[Hoyos et al. (2019), 012005](https://iopscience.iop.org/article/10.1088/1742-6596/1247/1/012005)
and [Hoyos et al. (2019), 012006](https://iopscience.iop.org/article/10.1088/1742-6596/1247/1/012006).

Unlike the previous fluid models, the equation $Q_m(\bar\omega, \bar k)=0$ does not generally admit an analytical solution. Instead, the zeros of the dispersion function are located numerically,and the resulting complex frequencies determine both wave propagation and damping. Because the function becomes singular at $\bar k=0$, the continuation starts at finite $\bar k$ and uses the nearby warm-fluid branch as an initial frequency estimate.


In [ ]:
kinetic_mode = (0, 1)
x_kinetic = bessel_zero(*kinetic_mode)
kbar_kinetic = np.linspace(0.35, 8.0, 65)
k_kinetic = wavenumber_per_meter(kbar_kinetic)

kinetic_root_values: list[complex] = []
failed_kinetic_roots: list[float] = []
previous_root = (
    warm_frequency(kbar_kinetic[0], cbar, x_kinetic, lambda_D_bar, "low").item() - 1e-5j
)

for kb in kbar_kinetic:
    warm_guess = warm_frequency(kb, cbar, x_kinetic, lambda_D_bar, "low").item()
    initial_guess = previous_root if np.isfinite(previous_root) else warm_guess - 1e-5j
    try:
        root_value = kinetic_root(kb, cbar, x_kinetic, lambda_D_bar, initial_guess)
    except RuntimeError:
        failed_kinetic_roots.append(kb)
        root_value = np.nan + 1j * np.nan
    else:
        previous_root = root_value
    kinetic_root_values.append(root_value)

kinetic_roots = np.array(kinetic_root_values, dtype=complex)
warm_reference = warm_frequency(kbar_kinetic, cbar, x_kinetic, lambda_D_bar, "low")
cold_reference = cold_frequency(kbar_kinetic, cbar, x_kinetic, "low")

if failed_kinetic_roots:
    failed = ", ".join(f"{kb:.2f}" for kb in failed_kinetic_roots)
    msg = f"Kinetic root continuation failed at kbar = {failed}"
    raise RuntimeError(msg)

fig, axes = plt.subplots(ncols=2, figsize=(12, 4.6), sharex=True)
axes[0].plot(
    k_kinetic, frequency_hz(cold_reference), color="0.55", linestyle="--", label="cold"
)
axes[0].plot(
    k_kinetic,
    frequency_hz(warm_reference),
    color=COLORS[kinetic_mode],
    label="warm-fluid",
)
axes[0].plot(
    k_kinetic,
    frequency_hz(kinetic_roots.real),
    color="#CC79A7",
    label=r"kinetic $Q_m$ root",
)
axes[0].set_title(r"Real frequency for TM$_{0,1}$")
axes[0].set_xlabel(r"$k$ [m$^{-1}$]")
axes[0].set_ylabel("frequency [Hz]")
axes[0].legend(fontsize=9)

axes[1].plot(k_kinetic, frequency_hz(kinetic_roots.imag), color="#CC79A7")
axes[1].axhline(0.0, color="0.35", linewidth=1.0, linestyle="--")
axes[1].set_title(r"Imaginary frequency from $Q_m$")
axes[1].set_xlabel(r"$k$ [m$^{-1}$]")
axes[1].set_ylabel(r"Im$(\omega) / 2\pi$ [Hz]")

fig.tight_layout()
plt.show()

## Conclusions

The cylindrical geometry establishes a discrete spectrum of transverse magnetic modes, where the radial mode structure is determined by the Bessel eigenvalues $X_{mn}$ and the longitudinal wavenumber controls propagation along the plasma column. Comparing the empty waveguide and plasma filled cases shows how the presence of mobile charged particles introduces additional waves branches beyond the purely electromagnetic behavior of the bounded geometry, as a neutral gas would not support that low frequency branch in this model.

The cold-fluid model provides the simplest plasma description and serves as a baseline for understanding the coupled effects of geometry and collective plasma response. The warm-fluid model extends this picture by incorporating finite electron temperature through the Debye length with $3\bar\lambda_D^2\bar k^2$, the solutions show that the correction is small when
$k\lambda_D$ is small and becomes more visible at larger axial wavenumber $k$. Because both fluid models admit analytics solutions, they provide controlled benchmarks for validating the numerical root finding procedure used throughout the notebook.

The numerical solutions reproduce the analytical cold and warm-fluid dispersion relations with very small relative error, confirming that the numerical implementation accurately captures the underlying dispersion equations. This validation establishes confidence in the computational framework before applying it to models where analytical solutions are generally unavailable.

The model that we studied without analytical solution, the kinetic model, changes the problem by replacing the fluid closure with a full velocity space electron response. In this setting, resonant wave-particle interactions are naturally included through the plasma dispersion function, and the allowed mode frequencies become complex, allowing the description of collisionless damping through Landau damping. The previous analytical benchmarks were an essential validation step in order to solve this kinetic dispersion relation numerically.

Taken together, the progression from empty guide to fluid and kinetic models illustrates how increasingly complete plasma descriptions modify wave propagation in bounded geometries, while demonstrating how analytical theory and numerical methods complement one another when studying plasma dispersion phenomena.

## References

- N. A. Krall and A. W. Trivelpiece, *Principles of Plasma Physics*,
  McGraw-Hill, 1973. See section 4.8, "Space Charge Waves."
- A. W. Trivelpiece and R. W. Gould,
  ["Space Charge Waves in Cylindrical Plasma Columns"](https://authors.library.caltech.edu/records/hfhkd-mm296),
  *Journal of Applied Physics* **30**, 1784-1793 (1959).
- H. Hoyos, Jaime and A. Rodriguez, Carlos,
  ["Numerical Solution of an Eigenvalue Problem for Bounded Plasma"](http://www.scielo.org.co/scielo.php?script=sci_arttext&pid=S1692-33242011000200017),
  *Revista Ingeniería Universidad de Medellín* **10**(19), 179-188 (2011).
  ISSN 1692-3324.
- Jaime H. Hoyos et al.,
  ["Numerical solution of the Vlasov-Maxwell system of equations for the dispersion relation of transverse magnetic modes in a cylindrical waveguide filled with plasma"](https://iopscience.iop.org/article/10.1088/1742-6596/1247/1/012005),
  *Journal of Physics: Conference Series* **1247**, 012005 (2019).
- Jaime H. Hoyos et al.,
  ["Landau damping in cylindrical inhomogeneous plasmas"](https://iopscience.iop.org/article/10.1088/1742-6596/1247/1/012006),
  *Journal of Physics: Conference Series* **1247**, 012006 (2019).
- PlasmaPy documentation for
  [`plasma_frequency`](https://docs.plasmapy.org/en/stable/api/plasmapy.formulary.frequencies.plasma_frequency.html),
  [`Debye_length`](https://docs.plasmapy.org/en/stable/api/plasmapy.formulary.lengths.Debye_length.html), and
  [`plasma_dispersion_func_deriv`](https://docs.plasmapy.org/en/stable/api/plasmapy.dispersion.dispersion_functions.plasma_dispersion_func_deriv.html).
